In [1]:
import sys
sys.path.append('..')

In [2]:
import torch
import torch.nn as N
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from src.utils.config import preprocessed_train_data,preprocessed_test_data
from src.utils.config import cat_cl
from models.model import ChurnMLP
import json


Loading data from data mkdir

In [3]:
train_data=torch.load(preprocessed_train_data)
test_data=torch.load(preprocessed_test_data)


Get categorical(embeddings) , numerical features and targets from train & test

In [4]:
X_train_num, X_train_cat =train_data['X_num'],train_data['X_cat']
X_test_num, X_test_cat =test_data['X_num'],test_data['X_cat']

traintarg, testtarg = TensorDataset(train_data['target']),TensorDataset(test_data['target'])

Initialize DataLoader for fututre Batches and dictionary with categories and their classes

In [5]:
X_train=DataLoader(TensorDataset(torch.cat((X_train_num,X_train_cat),dim=1)),batch_size=64,shuffle=True)

In [6]:
X_test=DataLoader(TensorDataset(torch.cat((X_test_num,X_test_cat),dim=1)),batch_size=256,shuffle=False)

In [7]:
with open(cat_cl,'r') as file:
    cat_classes=json.load(file)

Initialize model, pos_weight and optimizer (I'll choose Adam)

In [8]:
churn_predictor=ChurnMLP(num_lay_sz=X_train_num.shape[-1],cat_classes=cat_classes,emb_dim=8)

In [10]:
optimizer=optim.Adam(churn_predictor.parameters(),lr=1e-3)

In [20]:
pos_weight=len(train_data['target'][train_data['target']==0])/len(train_data['target'][train_data['target']==1])
pos_weight

2.768561872909699

Initalize BCEWithLogitsLoss

In [21]:
loss=N.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))